In [3]:
import sys
sys.path.append('../')
import numpy as np
import matplotlib.pyplot as plt
from qiskit_nature.second_q.hamiltonians.lattices import (
    KagomeLattice,
    BoundaryCondition,
)
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit_aer import AerSimulator


In [4]:
Lx, Ly = 3, 2
kagome = KagomeLattice(rows=Lx, cols=Ly,
    boundary_condition=(BoundaryCondition.PERIODIC, BoundaryCondition.PERIODIC)
)
nq = kagome.num_nodes
weighted_edges = kagome.weighted_edge_list
nlist = [(i, j) for i, j, _ in weighted_edges if i != j]

In [5]:
hc = 3.044382
h = hc
hzz = SparsePauliOp.from_sparse_list([("ZZ", pair, 1.0) for pair in nlist], num_qubits=nq)
hx = SparsePauliOp.from_sparse_list([("X", [i], h) for i in range(nq)], num_qubits=nq)
h_tot = hzz + hx
obs = SparsePauliOp.from_sparse_list([("Z", [(nq-1)//2], 1.0)], num_qubits=nq)
(nq-1)//2

8

In [4]:
t = 1
dt = 0.02
n_steps = int(t/dt)

qc = QuantumCircuit(nq)

# Save the initial state (t=0)
qc.save_statevector(label='psi_0')

for step in range(n_steps):
    # Append time evolution for one step dt
    evo = PauliEvolutionGate(h_tot, time=dt, synthesis=LieTrotter())
    qc.append(evo, range(nq))
    # Save the state after this step
    qc.save_statevector(label=f'psi_{step+1}')

In [ ]:
sim = AerSimulator(method='statevector')
qc_compiled = transpile(qc, sim)
result = sim.run(qc_compiled).result()
data = result.data(0)

In [ ]:
# Extract expectation values
times = np.arange(n_steps + 1) * dt
exp_vals = []
for step in range(n_steps + 1):
    sv = Statevector(data[f'psi_{step}'])
    exp_val = sv.expectation_value(obs).real
    exp_vals.append(exp_val)

In [ ]:
np.save('kagome_qiskit.npy', exp_vals)

In [ ]:
# # Plot
# plt.plot(times, exp_vals, 'o-', markersize=3)
# plt.grid(True)
# plt.show()